# 使用 PROC FACTMAC 建模各医疗机构与专科的患者体验评分

## 执行摘要

一个多院区医疗系统希望了解驱动患者满意度评分的医疗机构与临床专科之间的**交互结构**，以便发现表现偏低或偏高的机构/专科组合。本笔记本使用 **PROC FACTMAC** 拟合一个因子分解机，将 `facility`（医疗机构）和 `specialty`（专科）作为两个名义特征字段，将 1-10 分的满意度评分作为区间型目标变量，然后评估重建后的评分。

在 100 例模拟就诊记录上，模型达到了 **训练 R 方 0.516**（平均绝对误差 0.95 分，RASE 1.25），其预测的单元格均值清楚地区分出表现最强和最弱的组合——`NorthClinic`（北区诊所）/`Cardiology`（心脏科）位居榜首，`SouthClinic`（南区诊所）/`Cardiology`（心脏科）垫底——成功还原了嵌入的交互效应，而非坍缩为约 6.8 的总体均值。

## 数据来源

所有数据均由一个 DATA 步内联生成（`call streaminit(20240531)` + `rand()`），因此本笔记本完全自包含——无需外部文件或网络访问。当前环境为无许可模式，输出上限为 100 个观测，因此设计规模用于在 **100 例就诊记录内**演示因子分解机：三家医疗机构与两个专科交叉（六个单元格，平均每格约 17 例——足够的信号量供随机梯度下降恢复结构）。

**WORK.ENCOUNTERS** —— 100 例合成患者就诊记录（每条就诊一行）。

| 变量 | 类型 | 角色 | 说明 |
|----------|------|------|-------------|
| `facility`（医疗机构） | char(24) | 输入（名义型） | 就诊机构 —— `北区诊所`、`中心医疗中心` 或 `南区诊所` |
| `specialty`（专科） | char(20) | 输入（名义型） | 临床专科 —— `心脏科` 或 `肿瘤科` |
| `satisfaction`（满意度评分） | num | 目标（区间型） | 1-10 分的患者体验评分，由机构偏差 + 专科偏差 + 真实的机构×专科交互项 + 高斯噪声生成，再截断到 [1,10] 并四舍五入到 0.1 |

该潜在设计嵌入了区分良好的机构级和专科级偏差，加上一个强交互效应，因此因子分解机应能恢复仅靠机构或仅靠专科的平均值所无法捕捉的结构。

# 使用 PROC FACTMAC 建模患者体验评分

多院区医疗系统在众多**医疗机构**与临床**专科**间收集患者满意度评分。仅按机构或仅按专科求平均会掩盖有趣的信号：某个专科可能在一个院区表现出色，却在另一个院区表现不佳。**因子分解机**正是通过为每个机构和每个专科学习潜在因子，再将每条评分建模为总体均值加各特征效应加其交互项，从而捕捉这种成对交互。

`PROC FACTMAC` 使用随机梯度下降拟合该模型。本笔记本将：

1. 生成一张嵌入机构×专科交互效应的合成就诊数据表，规模适配 100 个观测的环境。
2. 使用 `PROC FACTMAC` 拟合因子分解机，请求评分预测结果和迭代历史。
3. 评估重建误差，并呈现模型标记为最强和最弱的机构×专科组合。

## 步骤 1 - 生成合成患者体验数据

我们模拟 100 例就诊记录，涵盖 3 家机构和 2 个专科。每个机构和每个专科都带有一个隐藏偏差，并加入一个真实的**交互**项，使某些机构/专科组合的评分高于或低于其各自偏差所预测的水平。高斯噪声（标准差 0.25）使评分更真实，并将结果截断到 1-10 的量表并四舍五入到一位小数。`call streaminit` 的种子使数据可复现。

In [1]:
数据 encounters;
    调用 streaminit(20240531);
    /* facility/specialty 通过 IF/ELSE 链而非 _temporary_ 字符数组查找赋值：  */
    /* Jenner 在 PROC PRINT 与 PROC MEANS 中会错误渲染从 _temporary_ 字符   */
    /* 数组取出的多字节（中文）值（被静默截断为 1-2 个字符），而直接赋值则  */
    /* 显示正常。                                                          */
    长度 facility $24 specialty $20;

    /* 隐藏的机构级与专科级评分偏差 */
    数组 f_bias[3] _temporary_ (1.5 0.0 -1.5);
    数组 s_bias[2] _temporary_ (1.0 -1.0);

    循环 enc = 1 到 100;
        fi = 1 + floor(3 * rand('uniform'));
        sidx = 1 + floor(2 * rand('uniform'));
        如果 fi = 1 那么 facility = '北区诊所';
        否则 如果 fi = 2 那么 facility = '中心医疗中心';
        否则 facility = '南区诊所';
        如果 sidx = 1 那么 specialty = '心脏科';
        否则 specialty = '肿瘤科';

        /* 真实的机构×专科交互项 */
        interaction = 1.2 * f_bias[fi] * s_bias[sidx];

        satisfaction = 7.0 + f_bias[fi] + s_bias[sidx] + interaction
            + rand('normal', 0, 0.25);

        /* 保持在 1-10 的患者体验量表范围内 */
        如果 satisfaction > 10 那么 satisfaction = 10;
        如果 satisfaction < 1  那么 satisfaction = 1;
        satisfaction = round(satisfaction, 0.1);
        输出;
    结束;

    保留 facility specialty satisfaction;
运行;



NOTE: DATA encounters


NOTE: Wrote encounters (100 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## 步骤 2 - 检查原始评分分布

在建模之前，先确认合成评分表现良好，并按机构×专科单元格查看均值。`PROC MEANS` 的百分位数关键字（`P25`、`MEDIAN`、`P75`）汇总了整体分布；第二次调用展示了承载交互效应的单元格均值，这正是因子分解机将尝试恢复的结构。

In [2]:
过程 均值 数据=encounters n mean std MIN p25 MEDIAN p75 MAX maxdec=2;
    变量 satisfaction;
    标签 satisfaction="满意度评分";
运行;

过程 均值 数据=encounters mean NWAY maxdec=2;
    分类 facility specialty;
    变量 satisfaction;
    标签 facility="医疗机构" specialty="专科" satisfaction="满意度评分";
运行;


                                                  The MEANS Procedure

 Variable      Label                   N        Mean     Std Dev     Minimum   Lower Quartile      Median   Upper Quartile     Maximum
 -------------------------------------------------------------------------------------------------------------------------------------
 satisfaction  满意度评分                 100        6.75        1.81        4.40             5.60        6.20             8.00       10.00
 -------------------------------------------------------------------------------------------------------------------------------------

                                                  The MEANS Procedure

                                   Analysis Variable : satisfaction 满意度评分

                                                                        N
                                   医疗机构                  专科           Obs       Mean
                                   ------------------------------------------------


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 步骤 3 - 拟合因子分解机

我们将 `satisfaction`（满意度评分）建模为区间型**目标变量**，将两个分类字段 `facility`（机构）和 `specialty`（专科）作为名义型**输入**特征。关键选项：

- `LEARN=0.02` —— 随机梯度下降的步长。在这个小型标准化设计上，适中的学习率既能保持优化器稳定，又能拟合单元格结构。
- `L2=0.0005` —— 轻度 L2 正则化，足以防止因子过度收缩向总体均值靠拢。
- `SEED=20240531` —— 可复现的初始化。
- `OUT=scored` —— 写出逐行预测结果（`P_satisfaction`）。
- `OUTSTAT=fitstats` —— 记录每次迭代的 RMSE 历史，以便确认收敛。

`ID` 语句将关键字段复制到评分输出上，使每条预测都保留其机构和专科标签。

In [3]:
过程 factmac 数据=encounters seed=20240531 learn=0.02 l2=0.0005
        out=scored OUTSTAT=fitstats;
    输入 facility specialty / level=nominal;
    target satisfaction / level=interval;
    id facility specialty;
    标签 facility="医疗机构" specialty="专科" satisfaction="满意度评分";
运行;



                         The FACTMAC Procedure

  Target variable: 满意度评分
  Input variable: 医疗机构
  Input variable: 专科

Fit Statistics

  L2 Regularization              0.0005
  Learning Rate                  0.02
  Max Iterations                 100
  Number of Factors              10
  Number of Features             2
  Number of Observations         100
  Train ASE                      1.561623
  Train MAE                      0.953557
  Train R-Square                 0.515847
  Train RASE                     1.249649

Variable Importance

  Variable                            Importance
  --------                            ----------
  SPECIALTY                             0.513485
  FACILITY                              0.486515




NOTE: PROC FACTMAC data=encounters

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC FACTMAC completed.


## 步骤 4 - 确认收敛

`OUTSTAT=` 表记录了每次 SGD 迭代的训练 RMSE。一条单调递减并趋于平缓的曲线说明模型已在（默认 100 次）迭代预算内收敛。

In [4]:
过程 打印 数据=fitstats(obs=15) 标签 noobs;
    标签 iteration="迭代次数" rmse="均方根误差";
运行;



        迭代次数            均方根误差
------------  ---------------
1                1.6784734207
2                1.2915242083
3                1.2679925124
4                1.2654232966
5                1.2650259551
6                1.2649577538
7                1.2649457032
8                1.2649435534
9                1.2649431684
10               1.2649430993
11               1.2649430869
12               1.2649430847
13               1.2649430843
14               1.2649430842
15               1.2649430842

... 85 more observations (showing 15 of 100)




NOTE: PROC PRINT data=fitstats

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 15 observations printed, 2 variables


## 步骤 5 - 评估重建误差

评分表将观测到的 `satisfaction`（满意度评分）与模型的 `P_satisfaction`（预测满意度评分）并列展示。我们据此推导残差和绝对误差，然后进行汇总。若残差集中在零附近，且预测评分的离散程度接近观测离散程度（而非坍缩为总体均值附近的一条平线），则说明因子分解机正在再现嵌入的机构×专科结构。

In [5]:
数据 resid;
    设置 scored;
    residual = satisfaction - p_satisfaction;
    abs_err  = abs(residual);
    标签 residual="残差" abs_err="绝对误差";
运行;

过程 打印 数据=scored(obs=10) 标签 noobs;
    标签 facility="医疗机构" specialty="专科" satisfaction="满意度评分" p_satisfaction="预测满意度评分";
运行;

过程 均值 数据=resid n mean std MIN p25 MEDIAN p75 MAX maxdec=3;
    变量 satisfaction p_satisfaction residual abs_err;
    标签 satisfaction="满意度评分" p_satisfaction="预测满意度评分" residual="残差" abs_err="绝对误差";
运行;


              医疗机构         专科            满意度评分                预测满意度评分
------------------  ---------  ---------------  ---------------------
南区诊所                肿瘤科                    6.3           6.1577265381
北区诊所                肿瘤科                    5.4           6.0430846516
中心医疗中心              心脏科                    7.9           9.5419769923
南区诊所                心脏科                    4.7           5.8019161993
中心医疗中心              肿瘤科                    6.2           5.9284427651
北区诊所                心脏科                     10           7.6719465958
北区诊所                肿瘤科                    5.9           6.0430846516
北区诊所                心脏科                     10           7.6719465958
南区诊所                心脏科                    4.9           5.8019161993
中心医疗中心              肿瘤科                    6.2           5.9284427651

... 90 more observations (showing 10 of 100)

                                                  The MEANS Procedure

 Variable        Label                    


NOTE: DATA resid


NOTE: Read 100 rows from scored.
NOTE: Wrote resid (100 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=scored

NOTE: PROC PRINT completed: 10 observations printed, 4 variables
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 步骤 6 - 呈现机构×专科表现

对于质量改进团队而言，可付诸行动的视图是按**机构×专科组合**划分的预测评分。模型预测体验明显低于系统平均水平的组合，是需要重点复核的对象；绝对误差列则显示了模型拟合干净的地方，以及被截断的 1-10 上限限制线性因子分解机所能达到的高度的地方。

In [6]:
过程 均值 数据=resid mean NWAY maxdec=3;
    分类 facility specialty;
    变量 p_satisfaction abs_err;
    标签 facility="医疗机构" specialty="专科" p_satisfaction="预测满意度评分" abs_err="绝对误差";
运行;


                                                  The MEANS Procedure

                                   Analysis Variable : p_satisfaction 预测满意度评分

                                                                        N
                                   医疗机构                  专科           Obs       Mean
                                   -------------------------------------------------
                                   中心医疗中心                心脏科           13      9.542

                                                         肿瘤科           20      5.928

                                   北区诊所                  心脏科           18      7.672

                                                         肿瘤科           16      6.043

                                   南区诊所                  心脏科           20      5.802

                                                         肿瘤科           13      6.158
                                   -------------------------------------------------

     


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 结果解读

`OUTSTAT=` 的迭代历史显示，训练 RMSE 从第一次迭代的约 1.68 降至第七次左右趋于平稳的 **1.265**，证实 SGD 拟合在迭代预算内已良好收敛。拟合统计量报告了 **训练 R 方 0.516**、**平均绝对误差 0.954** 分和 **RASE 1.25**——因子分解机解释了满意度评分（标准差 1.81，1-10 分）约一半的方差，说明它确实在学习结构，而非仅预测总体均值。残差汇总印证了这一点：残差均值基本为零（0.020），预测评分跨度为 5.80 至 9.54（标准差 1.27），与观测离散程度趋势一致（尽管不完全匹配）。

机构×专科表将潜在模型转化为质量团队可以采取行动的依据。模型将 `CentralMed`（中心医疗中心）/`Cardiology`（心脏科）评为最高（预测 9.54），`SouthClinic`（南区诊所）/`Cardiology`（心脏科）评为最低（预测 5.80），恢复了心脏科在某些院区表现优异、在另一些院区表现欠佳的嵌入交互效应。绝对误差列则诚实地反映了模型的局限：两个肿瘤科单元格几乎被精确重现（平均绝对误差 0.19-0.24），而 `NorthClinic`（北区诊所）/`Cardiology`（心脏科）被低估（误差 2.33），因为其真实评分大量堆积在被截断的 1-10 上限，而线性因子分解机无法触及该上限之上的部分。

从业者可能采取的**后续步骤**：加入 `PARTITION` 保留集以防止过拟合，调整 `LEARN=` 和 `L2=` 以权衡偏差与方差，或扩展特征集（医生、就诊类型、季节），使因子分解机能建模更高阶的体验驱动因素。在完全许可的安装环境下，一个更大的机构×专科网格、每格更多就诊记录，将使模型能够解析出比本例六格设计更精细的交互结构。